<a href="https://colab.research.google.com/github/Zeshan811/StarterNotebookA1/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip -q install duckdb huggingface_hub scikit-learn

import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'fact_daily': f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}
print("Connected.")

Paste your Hugging Face READ token (hf_...): ··········
Connected.


In [2]:
feature_frame = con.sql(f"""
    SELECT
        content_hash_id,
        SUM(gsc_impressions) AS total_impressions,
        SUM(gsc_clicks) AS total_clicks,
        AVG(gsc_avg_position) AS avg_position_month,
        (SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0)) AS ctr_month
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01' AND report_date < DATE '2026-04-01'
    GROUP BY content_hash_id
    HAVING SUM(gsc_impressions) >= 100
""").df()

print(feature_frame.shape)
feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(101441, 5)


,content_hash_id,total_impressions,total_clicks,avg_position_month,ctr_month
0,content_7a105f548d9c6916,6523.0,7.0,7.209549,0.001073
1,content_a3ea9792f793ec72,453.0,0.0,2.987198,0.000000
2,content_36c36abc7650d7af,5630.0,6.0,6.724039,0.001066
3,content_a7da352b73b02668,4944.0,13.0,7.244844,0.002629
4,content_1855a661b4d36130,429.0,1.0,4.209227,0.002331


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*



I'm using **K-Means clustering** — the natural fit for my lane (Structured
Content Archetype Clustering). My question is "what natural groups of
pages exist?", not "predict a label" — so a supervised method (logistic
regression, decision tree, etc.) doesn't apply; there's nothing to predict.

Why K-Means specifically over alternatives:
- Hierarchical clustering doesn't scale well to ~176K content items.
- DBSCAN needs density-tuning that's hard to justify without domain
  knowledge of "how far apart" pages should be.
- K-Means is simple, fast at this scale, and its output (cluster centers)
  is directly interpretable — I can describe each cluster by its average
  feature values, which matches the lane's goal of naming archetypes and
  mapping them to actions.

Features used: total_impressions, total_clicks, avg_position_month,
ctr_month — the same GSC-based, leakage-free features confirmed in my
data contract (ML-04).

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*



Clustering is unsupervised, so there's no label to hold out — but "no
split at all" would be dishonest, since a clustering result could just be
overfit noise. I use two honest checks instead of a train/test split:

1. **Stability split:** fit K-Means on a random 70% of content items, then
   assign the remaining 30% to their nearest fitted cluster and compare
   silhouette scores. If clusters only "look good" on the exact rows they
   were fit on, the pattern isn't real.
2. **Feature scaling:** all features are standardized (mean 0, std 1)
   before clustering, so no single large-scale feature (like impressions,
   which can be in the hundreds of thousands) silently dominates the
   distance calculation.

A client-grouped split isn't the right tool here, since my unit of
analysis is already one row per content item (not per client), and
content items aren't shared across clients in this table.

In [3]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

features = ["total_impressions", "total_clicks", "avg_position_month", "ctr_month"]
X = feature_frame[features].fillna(0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test = train_test_split(X_scaled, test_size=0.3, random_state=42)

print("Train shape:", X_train.shape, "| Test shape:", X_test.shape)

Train shape: (71008, 4) | Test shape: (30433, 4)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*



My Week-4 baseline was a simple manual rule: flag pages by a single
threshold on CTR-gap-from-tier-average, with one reason code. To compare
fairly on the SAME data and a comparable metric, I treat that baseline as
a simple "1-feature split" and compare it to K-Means's silhouette score
using all 4 features together — this tests whether the extra complexity
(multiple features, discovered rather than hand-picked groups) actually
earns its keep over a naive single-threshold split.

In [4]:
# Naive baseline: split into 2 groups purely by median CTR (mirrors the Week-4 rule's spirit)
median_ctr = feature_frame["ctr_month"].median()
naive_labels = (feature_frame["ctr_month"] > median_ctr).astype(int).values

naive_silhouette = silhouette_score(X_scaled, naive_labels)
print(f"NAIVE baseline (single CTR threshold, 2 groups): silhouette = {naive_silhouette:.3f}")

NAIVE baseline (single CTR threshold, 2 groups): silhouette = 0.205


In [5]:
# Try a few k values on the training set to pick a reasonable k
for k in [3, 4, 5, 6]:
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_train)
    sil = silhouette_score(X_train, km.labels_)
    print(f"k={k}: train silhouette = {sil:.3f}")

k=3: train silhouette = 0.441
k=4: train silhouette = 0.471
k=5: train silhouette = 0.473
k=6: train silhouette = 0.486


In [6]:
FINAL_K = 6

kmeans = KMeans(n_clusters=FINAL_K, random_state=42, n_init=10).fit(X_train)

train_silhouette = silhouette_score(X_train, kmeans.labels_)
test_labels = kmeans.predict(X_test)
test_silhouette = silhouette_score(X_test, test_labels)

print(f"K-MEANS (k={FINAL_K}), TRAIN silhouette: {train_silhouette:.3f}")
print(f"K-MEANS (k={FINAL_K}), TEST silhouette (stability check): {test_silhouette:.3f}")
print(f"\nNAIVE baseline silhouette (for comparison): {naive_silhouette:.3f}")

K-MEANS (k=6), TRAIN silhouette: 0.486
K-MEANS (k=6), TEST silhouette (stability check): 0.480

NAIVE baseline silhouette (for comparison): 0.205


| Method | Silhouette (train) | Silhouette (test/holdout) |
|---|---|---|
| Naive baseline (1 feature, 2 groups) | — | 0.205 |
| K-Means (4 features, k=6) | 0.486 | 0.480 |


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [7]:
final_kmeans = KMeans(n_clusters=FINAL_K, random_state=42, n_init=10).fit(X_scaled)
feature_frame["cluster"] = final_kmeans.labels_

cluster_profiles = feature_frame.groupby("cluster")[features].mean().round(3)
cluster_profiles["count"] = feature_frame.groupby("cluster").size()
print(cluster_profiles)

         total_impressions  total_clicks  avg_position_month  ctr_month  count
cluster                                                                       
0                25349.246        75.073              10.017      0.004   3413
1                 1921.490         4.126               9.256      0.002  70853
2                 1580.770         1.400              40.420      0.001  17031
3                 1590.106        16.524               8.550      0.012   9999
4                88174.146       500.396               5.760      0.008    144
5               617124.000      5668.000               2.383      0.009      1


In [8]:
from sklearn.metrics import silhouette_samples

sample_scores = silhouette_samples(X_scaled, feature_frame["cluster"])
feature_frame["silhouette_score"] = sample_scores

print("Lowest-confidence (most ambiguous) cluster assignments:")
feature_frame.sort_values("silhouette_score").head(10)[
    ["content_hash_id", "cluster", "silhouette_score"] + features
]

Lowest-confidence (most ambiguous) cluster assignments:


,content_hash_id,cluster,silhouette_score,total_impressions,total_clicks,avg_position_month,ctr_month
16923,content_5a431552d5138148,0,-0.238156,12769.0,45.0,12.373734,0.003524
93011,content_1f81488e5c8e960a,0,-0.232713,14373.0,34.0,12.638084,0.002366
18151,content_c729fc7e0e6fd484,0,-0.230986,13309.0,42.0,7.295041,0.003156
51326,content_da6d0a1f909b261a,0,-0.230164,13429.0,41.0,4.697355,0.003053
25216,content_f7f21fbe22989762,0,-0.229681,12557.0,47.0,5.405531,0.003743
10930,content_008670b743cac6c7,0,-0.229360,11905.0,51.0,16.372596,0.004284
78866,content_72761d1decf4ad42,0,-0.229066,13569.0,40.0,3.487675,0.002948
53042,content_8fd9c6d3e05b031a,4,-0.227902,61899.0,278.0,4.877347,0.004491
2718,content_d6012c6b6ae7e2ed,0,-0.226269,12740.0,46.0,3.368062,0.003611
94654,content_d59f8d17eb443cac,0,-0.225817,13350.0,42.0,4.071825,0.003146


**Cluster interpretation:**

- Cluster 0 (n=3,413) — "Mid-tier visible": moderate impressions (25K),
  moderate CTR, decent position (~10). Steady performers, not urgent.
- Cluster 1 (n=70,853) — "Background/low-signal": the largest group by far
  — low impressions, low clicks, low CTR. Likely low-demand content;
  action: monitor, not priority for review.
- Cluster 2 (n=17,031) — "Buried pages": very deep average position (40.4)
  — essentially invisible in search. Low CTR here reflects position, not
  a title/meta problem. Action: different lever needed (visibility/ranking
  work, not a CTR fix).
- Cluster 3 (n=9,999) — "Hidden gems": modest impressions but the highest
  CTR (0.012) of any normal cluster — these pages convert well relative to
  their traffic. Action: consider promoting/expanding, they're proven to
  work when seen.
- Cluster 4 (n=144) — "Champions": high impressions, high clicks, strong
  position (5.76), solid CTR. Action: protect, don't disturb.
- Cluster 5 (n=1) — a single extreme-traffic page, likely a homepage or
  a viral/seasonal spike. Too small to be a real archetype — this is a
  K-Means artifact from an outlier, not a meaningful group.

**Where the model is "wrong" / a real limitation:**
K-Means is sensitive to extreme outliers — one page with unusually high
traffic pulled its own cluster (n=1), which isn't a useful archetype for
action planning. A production version should either cap/log-transform
impressions before clustering, or explicitly separate outliers before
running K-Means, so one page doesn't distort the group boundaries for
everyone else.

**What the model leans on:**
avg_position and total_impressions appear to drive most of the separation
— cluster 2 splits almost entirely on position (buried vs not), while
clusters 1, 3, and 4 separate mostly on impression/click scale. CTR adds
a meaningful third dimension mainly for distinguishing cluster 3 ("hidden
gems") from cluster 1 ("background").

**Does complexity earn its keep?**
Compared to the naive 1-feature baseline, K-Means with 4 features finds
richer structure — particularly cluster 2 (buried pages) and cluster 3
(hidden gems), which a single CTR threshold would never separate, since
both involve position and CTR interacting differently. This suggests the
extra complexity is worthwhile, though the outlier-sensitivity issue
(cluster 5) shows the method isn't perfect out of the box.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.